# Homework: Evaluation

## Setup

In [ ]:
import json
from tqdm.auto import tqdm
from evaluation.evaluation import generate_ground_truth, calc_total_price
import openai
import os
from models.question import Questions

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

In [4]:
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [5]:
doc = documents[0]
user_prompt = json.dumps(doc)
type(user_prompt)

str

### Generating ground truth

In [6]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions

In [7]:
docs = []
for i in range(3):
    doc = {
        "filename": documents[i]["filename"],
        "content": documents[i]["content"],
    }
    docs.append(doc)

print(docs)

[{'filename': '01-agentic-rag/lessons/01-intro.md', 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is th

In [8]:
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
model = os.environ["OPENAI_LLM_MODEL"]

In [9]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(docs):
    records, usage = generate_ground_truth(
        doc, client, data_gen_instructions, output_type=Questions, model=model
    )
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

```bash
[ResponseUsage(input_tokens=1023, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=106, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1129),
 ResponseUsage(input_tokens=1289, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=76, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1365),
 ResponseUsage(input_tokens=1756, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1865)]

```


In [10]:
avg_tokens = sum([usage.total_tokens for usage in usages]) / len(usages)
avg_tokens

1444.0

In [11]:
usages

[ResponseUsage(input_tokens=1023, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=98, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1121),
 ResponseUsage(input_tokens=1289, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=78, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1367),
 ResponseUsage(input_tokens=1756, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=88, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1844)]

In [12]:
with (
    open("./results/ground_truth.json", "w", encoding="utf-8") as f1,
    open("./results/usage.json", "w", encoding="utf-8") as f2,
):
    json.dump(ground_truth, f1, indent=4)
    # json.dump(usages, f2, indent=4)

In [13]:
total_cost = calc_total_price(
    usages, input_price_per_million=0.4, output_price_per_million=0.60
)
total_cost

0.0017856